## Example of Med2Vec

#### **Taken from Medium (https://medium.com/codex/med2vec-transforming-healthcare-with-data-driven-insights-987120317e80), authored by Everton Gomede**


In [5]:
# Import packages and modules:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, TimeDistributed
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, confusion_matrix
import tensorflow as tf

In [6]:
# Enable eager execution

tf.config.run_functions_eagerly(True)

# Note:  what is eager execution in tensorflow?
# https://jonathan-hui.medium.com/tensorflow-eager-execution-v-s-graph-tf-function-6edaa870b1f1

In [ ]:
# Generate synthetic dataset
def generate_synthetic_data(num_patients=1000, num_visits=10, num_codes=50):
    np.random.seed(42)
    data = []
    for _ in range(num_patients):
        patient_visits = []
        for _ in range(np.random.randint(1, num_visits+1)):
            visit = np.random.choice([f'code{i}' for i in range(1, num_codes+1)],
                                     np.random.randint(1, 5), replace=False).tolist()
            patient_visits.append(visit)
        data.append(patient_visits)
    return data

data = generate_synthetic_data()

# added by Barry:

type(data)

In [10]:

# Flatten the data and fit the LabelEncoder
all_codes = [code for patient in data for visit in patient for code in visit]
le = LabelEncoder()
le.fit(all_codes)

# Transform the codes to integers and pad sequences
data_encoded = [[le.transform(visit).tolist() for visit in patient] for patient in data]
data_padded = [pad_sequences(patient, padding='post').tolist() for patient in data_encoded]

# Prepare input and target data
max_len = max(max(len(visit) for visit in patient) for patient in data_padded)
data_padded = [pad_sequences(patient, maxlen=max_len, padding='post').tolist() for patient in data_encoded]
input_data = [visit[:-1] for patient in data_padded for visit in patient if len(visit) > 1]
target_data = [visit[1:] for patient in data_padded for visit in patient if len(visit) > 1]

# Convert to numpy arrays
input_data = np.array(input_data)
target_data = np.expand_dims(np.array(target_data), -1)

# Split the data
X_train, X_val, y_train, y_val = train_test_split(input_data, target_data, test_size=0.2, random_state=42)

# Model parameters
vocab_size = len(le.classes_)
embedding_dim = 128
lstm_units = 128